# Clustering Track
23CSE301 ML Capstone — Review 2

**K-Means** and **Agglomerative Hierarchical Clustering** on the [Indian Vehicle Registration Data (2020–25)](https://www.kaggle.com/datasets/aatifahmad123/indian-vehicle-registration-data-202025) (Kaggle). There are no labels during training; any ground-truth labels are used **only afterwards** to interpret/validate the clusters (guidelines 3.3).

**Get the data first:** `python data/download_dataset.py vehicle-reg` (needs a free Kaggle API token) — the CSV(s) land in `data/raw/vehicle_registration/`. Sections A–B detect the file and its column types automatically, so no code edits are needed; if the folder holds several CSVs, set `DATA_FILE` below.

**Structure follows the Review 2 rubric:** A (Dataset & EDA) → B (Preprocessing) → C (K-Means, Agglomerative — *the algorithm cells are `TODO`*) → D (Evaluation metrics) → E (Cluster visualisation) → F (Post-hoc validation). Everything except the algorithm cells is pre-built and reads from `cluster_labels`, so keep that dict's keys exactly `"K-Means"` and `"Agglomerative"`.

Mandatory outputs: Elbow curve (K-Means), Dendrogram (Hierarchical), Silhouette / Davies-Bouldin / Calinski-Harabasz for **both** algorithms, PCA 2-D scatter for **every** algorithm, t-SNE for at least one.

## Section A — Dataset & EDA

### A1. Dataset Loading & Audit

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="colorblind")

DATA_DIR = Path("../data/raw/vehicle_registration")
DATA_FILE = None            # e.g. "registrations.csv" if the download contains several CSVs

# Filled in by the algorithm sections below; keys must be exactly "K-Means" and "Agglomerative"
cluster_labels = {}

csv_files = sorted(DATA_DIR.glob("**/*.csv")) if DATA_DIR.exists() else []
if DATA_FILE:
    csv_files = [DATA_DIR / DATA_FILE]
if csv_files:
    DATA_PATH = max(csv_files, key=lambda p: p.stat().st_size)      # largest CSV unless DATA_FILE is set
    print("CSV files found:", [p.name for p in csv_files], "\nUsing:", DATA_PATH.name)
    df = pd.read_csv(DATA_PATH)
else:
    df = None
    print("Dataset not found in", DATA_DIR, "— run:  python data/download_dataset.py vehicle-reg")

Dataset not found in ../data/raw/vehicle_registration — run:  python data/download_dataset.py vehicle-reg


In [2]:
if df is not None:
    print("Shape:", df.shape)
    df.info()
    display(df.head())
    miss = df.isna().sum()
    print("\nMissing values per column (non-zero only):")
    print(miss[miss > 0].sort_values(ascending=False) if (miss > 0).any() else "none")
    print("\nDuplicate rows:", df.duplicated().sum())
else:
    print("Skipped — no dataset loaded.")

Skipped — no dataset loaded.


### A2. EDA Visualisations

Distribution of every numeric feature, counts for the main categorical features, and a correlation heatmap. Add a Markdown observation after each major plot (A3).

In [3]:
if df is not None:
    num_cols = df.select_dtypes("number").columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]
    print(f"{len(num_cols)} numeric, {len(cat_cols)} non-numeric columns")

    shown = num_cols[:12]
    if shown:
        ncols = 3; nrows = int(np.ceil(len(shown) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        for ax, col in zip(axes.ravel(), shown):
            sns.histplot(df[col].dropna(), bins=40, ax=ax)
            ax.set_title(f"Distribution of {col}")
            ax.set_xlabel(col); ax.set_ylabel("Count")
        for ax in axes.ravel()[len(shown):]:
            ax.axis("off")
        plt.tight_layout(); plt.show()
else:
    print("Skipped — no dataset loaded.")

Skipped — no dataset loaded.


In [4]:
if df is not None:
    low_card = [c for c in cat_cols if 1 < df[c].nunique() <= 60][:6]
    if low_card:
        fig, axes = plt.subplots(len(low_card), 1, figsize=(10, 3.2 * len(low_card)), squeeze=False)
        for ax, col in zip(axes[:, 0], low_card):
            df[col].value_counts().head(10).plot(kind="bar", ax=ax)
            ax.set_title(f"Top categories — {col}")
            ax.set_xlabel(col); ax.set_ylabel("Records")
        plt.tight_layout(); plt.show()
    else:
        print("No low-cardinality categorical columns to plot.")
else:
    print("Skipped — no dataset loaded.")

Skipped — no dataset loaded.


In [5]:
if df is not None and len(num_cols) >= 2:
    corr_cols = num_cols[:15]
    plt.figure(figsize=(0.7 * len(corr_cols) + 4, 0.6 * len(corr_cols) + 3))
    sns.heatmap(df[corr_cols].corr(), annot=len(corr_cols) <= 12, fmt=".2f", cmap="coolwarm")
    plt.title("Correlation heatmap (numeric features)")
    plt.tight_layout(); plt.show()
else:
    print("Skipped — need a loaded dataset with at least two numeric columns.")

Skipped — need a loaded dataset with at least two numeric columns.


**Observation (write your own):** what do the distributions, category counts and correlations tell you about this data?

## Section B — Preprocessing

Clustering is distance-based, so features are cleaned and scaled first. The steps below are applied automatically and every decision is printed: drop duplicates → drop ID-like, constant and mostly-missing columns → impute (median / most frequent) → `log1p` for strongly right-skewed non-negative counts → one-hot encode low-cardinality categoricals → standardise. Agglomerative clustering needs O(n²) memory, so the data is **subsampled to at most `MAX_ROWS` rows** (fixed seed). Any known label column should be set aside as `y_true` (Section F) and kept out of `X_scaled`.

In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

MAX_ROWS = 4000          # Agglomerative clustering is O(n^2) in memory
MAX_ONEHOT_COLS = 150
y_true = None            # set to a label Series (aligned to df_model) if the dataset has ground truth

if df is not None:
    work = df.drop_duplicates().copy()
    log = []
    drop = []
    for col in work.columns:
        s = work[col]
        if s.isna().mean() > 0.5:
            drop.append((col, "more than 50% missing"))
        elif s.nunique(dropna=True) <= 1:
            drop.append((col, "constant"))
        elif str(col).lower().startswith("unnamed") or str(col).lower() in {"id", "index"}:
            drop.append((col, "index / ID column"))
        elif s.dtype == "object" and s.nunique() > 0.5 * len(work):
            drop.append((col, "ID-like (almost every value unique)"))
    work = work.drop(columns=[c for c, _ in drop])
    for c_, why in drop:
        log.append(f"dropped {c_!r}: {why}")

    if len(work) > MAX_ROWS:
        work = work.sample(MAX_ROWS, random_state=RANDOM_STATE)
        log.append(f"subsampled to {MAX_ROWS} rows")
    work = work.reset_index(drop=True)

    num_f = work.select_dtypes("number").columns.tolist()
    cat_f = [c for c in work.columns if c not in num_f]

    X_num = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(work[num_f]), columns=num_f) if num_f else pd.DataFrame(index=work.index)
    skewed = [c for c in num_f if (X_num[c] >= 0).all() and X_num[c].skew() > 2]
    for c_ in skewed:
        X_num[c_] = np.log1p(X_num[c_])
    if skewed:
        log.append(f"log1p applied to skewed counts: {skewed}")

    cat_keep, onehot_cols = [], 0
    for c_ in sorted(cat_f, key=lambda c: work[c].nunique()):
        k = work[c_].nunique()
        if k <= 40 and onehot_cols + k <= MAX_ONEHOT_COLS:
            cat_keep.append(c_); onehot_cols += k
        else:
            log.append(f"excluded categorical {c_!r} ({k} unique values — too many to one-hot)")
    parts = [X_num.values]
    feature_names = list(X_num.columns)
    if cat_keep:
        cat_imp = SimpleImputer(strategy="most_frequent").fit_transform(work[cat_keep])
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        parts.append(enc.fit_transform(cat_imp))
        feature_names += list(enc.get_feature_names_out(cat_keep))

    X_all = np.hstack(parts)
    X_scaled = StandardScaler().fit_transform(X_all)
    df_model = work                                  # original-valued rows aligned with X_scaled (used for cluster profiling)
    print("\n".join(log) if log else "no columns dropped")
    print("X_scaled shape:", X_scaled.shape, "| numeric:", len(num_f), "| one-hot categoricals:", cat_keep)
else:
    print("Skipped — no dataset loaded (X_scaled will exist once the data is downloaded).")

Skipped — no dataset loaded (X_scaled will exist once the data is downloaded).


**Preprocessing justification (write your own):** why were these cleaning / encoding / scaling decisions appropriate for this dataset?

## Section C — Clustering Algorithms

### C1. K-Means

Plot the **Elbow curve** (inertia vs. k) to choose k, then fit the final model.

In [7]:
# TODO: Elbow curve (inertia vs. k) — title, labelled axes required
# from sklearn.cluster import KMeans
# ks = range(2, 11)
# inertias = [KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled).inertia_ for k in ks]
# plt.plot(list(ks), inertias, marker="o"); plt.xlabel("k"); plt.ylabel("Inertia"); plt.title("K-Means elbow curve")
# plt.tight_layout(); plt.show()

In [8]:
# TODO: fit the final K-Means model and store its labels
# km = KMeans(n_clusters=..., random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
# cluster_labels["K-Means"] = km.labels_

**K-Means observation / choice of k (write your own):** TODO

### C2. Agglomerative Hierarchical Clustering

Plot the **Dendrogram** and compare linkage strategies (e.g. ward / complete / average / single), then fit the final model.

In [9]:
# TODO: Dendrogram (title, labelled axes required)
# from scipy.cluster.hierarchy import linkage, dendrogram
# Z = linkage(X_scaled, method="ward")
# plt.figure(figsize=(10, 5)); dendrogram(Z, truncate_mode="lastp", p=30)
# plt.xlabel("Cluster (size)"); plt.ylabel("Distance"); plt.title("Hierarchical clustering dendrogram"); plt.show()

In [10]:
# TODO: compare linkage strategies, then fit the final model and store its labels
# from sklearn.cluster import AgglomerativeClustering
# agg = AgglomerativeClustering(n_clusters=..., linkage="ward").fit(X_scaled)
# cluster_labels["Agglomerative"] = agg.labels_

**Hierarchical observation / linkage comparison (write your own):** TODO

## Section D — Evaluation Metrics

Silhouette Score, Davies-Bouldin Index and Calinski-Harabasz Index for **all** algorithms. Pre-built: fills in automatically once `cluster_labels` is populated.

In [11]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

if cluster_labels and "X_scaled" in globals():
    rows = []
    for name, labels in cluster_labels.items():
        rows.append({
            "Algorithm": name,
            "Clusters": len(set(labels)),
            "Silhouette (higher better)": silhouette_score(X_scaled, labels),
            "Davies-Bouldin (lower better)": davies_bouldin_score(X_scaled, labels),
            "Calinski-Harabasz (higher better)": calinski_harabasz_score(X_scaled, labels),
        })
    cluster_metrics = pd.DataFrame(rows)
    display(cluster_metrics)
else:
    print("Metrics appear once Section C fills cluster_labels.")

Metrics appear once Section C fills cluster_labels.


**Interpretation of the metrics (write your own):** TODO — which algorithm is better on each metric, and do the metrics agree?

## Section E — Cluster Visualisation

PCA-reduced 2-D scatter with cluster colour-coding for **every** algorithm, and a t-SNE plot for each algorithm. Pre-built; every plot has a title, labelled axes and a legend.

In [12]:
from sklearn.decomposition import PCA

if cluster_labels and "X_scaled" in globals():
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    pcs = pca.fit_transform(X_scaled)
    fig, axes = plt.subplots(1, len(cluster_labels), figsize=(6.5 * len(cluster_labels), 5), squeeze=False)
    for ax, (name, labels) in zip(axes[0], cluster_labels.items()):
        sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=labels, palette="tab10", s=15, ax=ax, legend="full")
        ax.set_title(f"PCA projection — {name}")
        ax.set_xlabel(f"Principal component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
        ax.set_ylabel(f"Principal component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
        ax.legend(title="Cluster", fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("PCA plots appear once Section C fills cluster_labels.")

PCA plots appear once Section C fills cluster_labels.


In [13]:
from sklearn.manifold import TSNE

if cluster_labels and "X_scaled" in globals():
    emb = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=min(30, len(X_scaled) - 1)).fit_transform(X_scaled)
    fig, axes = plt.subplots(1, len(cluster_labels), figsize=(6.5 * len(cluster_labels), 5), squeeze=False)
    for ax, (name, labels) in zip(axes[0], cluster_labels.items()):
        sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=labels, palette="tab10", s=15, ax=ax, legend="full")
        ax.set_title(f"t-SNE — {name}")
        ax.set_xlabel("t-SNE dimension 1"); ax.set_ylabel("t-SNE dimension 2")
        ax.legend(title="Cluster", fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("t-SNE plots appear once Section C fills cluster_labels.")

t-SNE plots appear once Section C fills cluster_labels.


**Observation (write your own):** TODO — are the clusters visually separated, and where do they overlap?

## Section F — Post-hoc Validation & Cluster Profiling

Only **after** clustering. Each cluster is profiled by its size and mean feature values (original units) to *interpret* it; if a ground-truth label `y_true` exists, Adjusted Rand Index, Normalised Mutual Information and a cross-tab are added. Labels are never used while fitting.

In [14]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

if cluster_labels and df is not None and "df_model" in globals():
    num_profile = df_model.select_dtypes("number").columns.tolist()
    for name, labels in cluster_labels.items():
        print(f"===== {name} — cluster profile (mean of numeric features) =====")
        prof = df_model[num_profile].groupby(np.asarray(labels)).mean().round(2)
        prof.insert(0, "size", pd.Series(np.asarray(labels)).value_counts().sort_index())
        display(prof)
        if y_true is not None:
            print(f"ARI = {adjusted_rand_score(y_true, labels):.4f} | NMI = {normalized_mutual_info_score(y_true, labels):.4f}")
            display(pd.crosstab(np.asarray(labels), np.asarray(y_true), rownames=["cluster"], colnames=["true label"]))
else:
    print("Cluster profiles appear once Section C fills cluster_labels.")

Cluster profiles appear once Section C fills cluster_labels.


**Post-hoc interpretation (write your own):** TODO — what distinguishes each cluster?

## Conclusion

TODO (write your own): problem → data → method → results → conclusion.